# Python 底盘运动控制

在本章节中我们会写一个Python例程，用于控制机器人底盘运动，你也可以自行使用其它语言来进行机器人底盘的运动控制。

## 底盘控制原理

在本例程中，我们使用 JupyterLab 中的代码块，生成一串 JSON 指令，通过树莓派的 GPIO 串口（默认与下位机通信的波特率为115200），将这个 JSON 指令发送给下位机，下位机收到指令后开始执行动作。

你可以参考后续的章节来了解都可以给下位机发送什么样的指令，你也可以使用其它语言来实现这一功能，或者自己写一个上位机的应用。

## 这样设计的优点

我们使用上位机+下位机的架构可以充分解放上位机的宝贵资源，上位机（树莓派，Jetson 等 SBC）类似人类的大脑，ESP32作为下位机类似人类的小脑，上位机执行视觉处理/决策方面的高阶控制，下位机执行具体的运动控制/插值等低阶控制。这样可以做到大小脑分工合作，下位机负责高频PID控制可保证车轮转速准确，上位机也不需要在这类低复杂度高算力的工作上浪费资源。

## 主程序 app.py

项目文件夹中的 app.py，这个是产品的主程序，当你执行过 autorun.sh 后(产品默认出厂是已经配置好自动运行的了)，app.py 会在未来每次开机时自动运行，它的运行会占用 GPIO串口 和 摄像头资源，如果你在交互式教程中或者其它程序中需要用到这些资源可能会引发冲突或其它错误，二次开发或学习前，务必关闭掉 app.py 的自动启动。

由于 app.py 中使用了多线程,且开机时它使用 service 来自动运行,所以通常不能使用 sudo killall python 这样的指令来关闭 app.py, 你需要执行以下命令，并且等待命令执行结束即可，无需重启机器人产品。

### 结束主程序
1. 点击上方本页面选项卡旁边的 “+”号，会打开一个新的名为 Launcher 的选项卡。
2. 点击 Other 内的 Terminal，打开终端窗口。
3. 在终端窗口内输入 `bash` 后按回车。
4. 现在你可以使用 Bash Shell 来控制机器人了。
5. 输入命令： `systemctl --user stop ugv-app.service`。
6. 在终端页面，命令执行完后，继续该教程的剩余部分。

注意：

如果需要永久关闭app.py 程序 ，则执行 systemctl --user disable ugv-app.service;

再次重启设备开机后产品主程序就不会自动运行了，你可以随意使用 JupyterLab 中的教程了。

后续如果你需要再恢复主程序开机自动运行时，执行 systemctl --user enable ugv-app.service，这样就能恢复主程序的开机自动运行了。

## 底盘控制例程

在下面的例程中，我们使用 is_raspberry_pi5() 函数来判断当前的树莓派型号，因为树莓派4B和树莓派5的 GPIO 串口的设备名称是不同的，你需要使用正确的 GPIO 设备名称，且使用与下位机相同的波特率（默认为115200）。

运行以下代码块之前你需要先将产品架高起来，保持驱动轮全部离地，调用以下代码块后机器人会开始走动，小心不要让机器人从桌面上掉落。

In [ ]:
from base_ctrl import BaseController
import time

# 用于检测树莓派的函数
def is_raspberry_pi5():
    with open('/proc/cpuinfo', 'r') as file:
        for line in file:
            if 'Model' in line:
                if 'Raspberry Pi 5' in line:
                    return True
                else:
                    return False

# 根据树莓派的型号来确定 GPIO 串口设备名称
if is_raspberry_pi5():
    base = BaseController('/dev/ttyAMA0', 115200)
else:
    base = BaseController('/dev/serial0', 115200)

# 以0.2m/s的速度转动2秒钟后停止
base.send_command({"T":1,"X":0.2,"Y":0.0,"Yaw":0.0})
time.sleep(2)
base.send_command({"T":1,"X":0.0,"Y":0.0,"Yaw":0.0})

通过调用上面的代码块，树莓派会首先发送 {"T":1,"X":0.2,"Y":0.0,"Yaw":0.0} 这条指令（后面章节我们会再具体介绍指令的构成），开始移动，间隔两秒钟后树莓派会发送 {"T":1,"X":0.0,"Y":0.0,"Yaw":0.0} 这条指令，会停止，这里需要注意的一点是，即使不发送后面的停止的指令，如果你没有发送新的指令，依然会停止转动，这是因为下位机内含有心跳函数，心跳函数的做用是在上位机长时间没有新的指令发送给下位机时，下位机自动停止目前的移动指令，改函数的目的是为了避免上位机由于某些原因死机而导致下位机继续运动。

如果你希望机器人一直持续不断地运动下去，上位机需要每隔2秒-4秒循环发送运动控制的指令。

## 移动原理

六足机器人的运动控制采用基于机身坐标系的速度指令方式。控制指令中，X 与 Y 分别表示机器人在机身前向及侧向方向的平移速度分量，Yaw 表示机器人绕机身垂直轴的角速度。当 Yaw 不为零时，机器人通过对各足端在世界坐标系中的目标轨迹叠加旋转分量，使支撑腿产生差速位移，从而形成绕机身中心的旋转运动，实现整机转向。
在转向过程中，各腿的步态时序保持不变，通过对支撑相位与抬腿相位分别进行运动补偿，保证足端与地面接触稳定，进而实现平稳、连续的原地或伴随平移的转向运动。